# Hyperparameter tuning by randomized search

In [1]:
import pandas as pd
adult_census = pd.read_csv("data/adult_data.csv")

In [3]:
target_name = "class"
target = adult_census[target_name]
target.head()

0    b'<=50K'
1    b'<=50K'
2     b'>50K'
3     b'>50K'
4    b'<=50K'
Name: class, dtype: object

In [6]:
data = adult_census.drop(columns=[target_name, "education-num"]) # education_num is redundant
data.head()

,age,workclass,fnlwgt,education,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,25.0,b'Private',226802.0,b'11th',b'Never-married',b'Machine-op-inspct',b'Own-child',b'Black',b'Male',0.0,0.0,40.0,b'United-States'
1,38.0,b'Private',89814.0,b'HS-grad',b'Married-civ-spouse',b'Farming-fishing',b'Husband',b'White',b'Male',0.0,0.0,50.0,b'United-States'
2,28.0,b'Local-gov',336951.0,b'Assoc-acdm',b'Married-civ-spouse',b'Protective-serv',b'Husband',b'White',b'Male',0.0,0.0,40.0,b'United-States'
3,44.0,b'Private',160323.0,b'Some-college',b'Married-civ-spouse',b'Machine-op-inspct',b'Husband',b'Black',b'Male',7688.0,0.0,40.0,b'United-States'
4,18.0,b'?',103497.0,b'Some-college',b'Never-married',b'?',b'Own-child',b'White',b'Female',0.0,0.0,30.0,b'United-States'


In [7]:
from sklearn.model_selection import train_test_split
data_train, data_test, target_train, target_test = train_test_split(
    data, target, random_state = 42
)

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import make_column_selector as selector

categorical_columns_selector = selector(dtype_include=object)
categorical_columns = categorical_columns_selector(data)

categorical_preprocessor = OrdinalEncoder(
    handle_unknown="use_encoded_value", unknown_value=-1
)
preprocessor = ColumnTransformer(
    [("cat_preprocessor", categorical_preprocessor, categorical_columns)],
    remainder="passthrough",
)

In [9]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline

model = Pipeline(
    [
        ("preprocessor", preprocessor),
        (
            "classifier",
            HistGradientBoostingClassifier(random_state=42, max_leaf_nodes=4),
        ),
    ]
)

model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat_preprocessor',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['workclass', 'education',
                                                   'marital-status',
                                                   'occupation', 'relationship',
                                                   'race', 'sex',
                                                   'native-country'])])),
                ('classifier',
                 HistGradientBoostingClassifier(max_leaf_nodes=4,
                                                random_state=42))])

`RandomSearchCV` is used similarly to the `GridSearchCV` but the sampling distributions need to be specified instead of the parameter values. For instance, we can draw candidates using a log-uniform distribution because the parameters we are interested in take positive values with a natural log scaling (.1 is as close to 1 as 10 is).

- Random search (with RandomizedSearchCV) is typically beneficial compared to grid search (with GridSearchCV) to optimize 3 or more hyperparameters.

## Parameters to be optimized
- `l2_regularization`: it corresponds to the strength of the regularization;
- `min_samples_leaf`: it corresponds to the minimum number of samples required in a leaf;
- `max_bins`: it corresponds to the maximum number of bins to construct the histograms.
- `learning_rate`: it corresponds to the speed at which the gradient-boosting corrects the residuals at each boosting iteration;
- `max_leaf_nodes`: it corresponds to the maximum number of leaves for each tree in the ensemble.

In [10]:
from scipy.stats import loguniform


class loguniform_int:
    """Integer valued version of the log-uniform distribution"""

    def __init__(self, a, b):
        self._distribution = loguniform(a, b)

    def rvs(self, *args, **kwargs):
        """Random variable sample"""
        return self._distribution.rvs(*args, **kwargs).astype(int)

In [11]:
%%time
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    "classifier__l2_regularization": loguniform(1e-6, 1e3),
    "classifier__learning_rate": loguniform(0.001, 10),
    "classifier__max_leaf_nodes": loguniform_int(2, 256),
    "classifier__min_samples_leaf": loguniform_int(1, 100),
    "classifier__max_bins": loguniform_int(2, 255),
}

model_random_search = RandomizedSearchCV(
    model,
    param_distributions=param_distributions,
    n_iter=10,
    cv=5,
    verbose=1,
)
model_random_search.fit(data_train, target_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
CPU times: total: 2min 54s
Wall time: 2min 6s


C:\Users\KIIT0001\miniconda3\Lib\site-packages\sklearn\compose\_column_transformer.py:1623: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(remainder='passthrough',
                                                                transformers=[('cat_preprocessor',
                                                                               OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                                              unknown_value=-1),
                                                                               ['workclass',
                                                                                'education',
                                                                                'marital-status',
                                                                                'occupation',
                                                                                'relationship',
                                                                                'race',
                                                                                'sex',
                                                                                'native-country'])])),
                                             ('classifier',
                                              HistGradientBoostingC...
                                        'classifier__learning_rate': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x000001EDD60DB710>,
                                        'classifier__max_bins': <__main__.loguniform_int object at 0x000001EDD60DBBF0>,
                                        'classifier__max_leaf_nodes': <__main__.loguniform_int object at 0x000001EDD60DBC50>,
                                        'classifier__min_samples_leaf': <__main__.loguniform_int object at 0x000001EDD60DBDD0>},
                   verbose=1)

In [12]:
accuracy = model_random_search.score(data_test, target_test)

print(f"The test accuracy score of the best model is {accuracy:.2f}")

The test accuracy score of the best model is 0.88


In [13]:
from pprint import pprint

print("The best parameters are:")
pprint(model_random_search.best_params_)

The best parameters are:
{'classifier__l2_regularization': 0.039427124800601926,
 'classifier__learning_rate': 0.9278055420143922,
 'classifier__max_bins': 149,
 'classifier__max_leaf_nodes': 3,
 'classifier__min_samples_leaf': 1}


In [14]:
# get the parameter names
column_results = [f"param_{name}" for name in param_distributions.keys()]
column_results += ["mean_test_score", "std_test_score", "rank_test_score"]

cv_results = pd.DataFrame(model_random_search.cv_results_)
cv_results = cv_results[column_results].sort_values(
    "mean_test_score", ascending=False
)


def shorten_param(param_name):
    if "__" in param_name:
        return param_name.rsplit("__", 1)[1]
    return param_name


cv_results = cv_results.rename(shorten_param, axis=1)
cv_results

,l2_regularization,learning_rate,max_leaf_nodes,min_samples_leaf,max_bins,mean_test_score,std_test_score,rank_test_score
4,0.039427,0.927806,3,1,149,0.866834,0.003050,1
6,0.000003,0.021034,33,5,18,0.851765,0.004180,2
3,0.009979,0.823463,212,38,142,0.844503,0.005525,3
2,0.000022,0.720165,63,74,6,0.826185,0.001688,4
9,9.589533,0.009774,122,64,7,0.815129,0.002919,5
0,1.889331,0.018974,6,11,23,0.813955,0.003710,6
5,0.000002,0.017624,51,30,4,0.810980,0.001658,7
1,0.000007,0.002753,8,1,3,0.758947,0.000013,8
7,0.001703,0.001435,32,1,11,0.758947,0.000013,8
8,0.019227,0.003602,8,19,4,0.758947,0.000013,8
